In [ ]:
import dataclasses
import numpy as np
import scipy.optimize
import matplotlib.ticker
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.visualization
import pandas
import named_arrays as na
import optika
import esis

In [ ]:
design = esis.flights.f2.optics.materials.multilayer_SiSc()

with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    design.plot_layers(
        ax=ax,
        thickness_substrate=20 * u.nm,
    )
    ax.set_axis_off()
    ax.autoscale_view()

In [ ]:
oxide, top, Sc, periodic = design.layers
Si_period, Sc_period = periodic.layers

period_design = Si_period.thickness + Sc_period.thickness
num_periods_design = periodic.num_periods + 1
fraction_Sc = (Sc_period.thickness / period_design).to(u.dimensionless_unscaled)
thickness_consumed = Si_period.thickness - top.thickness


def coating(
    num_periods: int,
    period: u.Quantity,
) -> optika.materials.MultilayerMirror:
    """
    The design, with the given number of periods and period.

    Parameters
    ----------
    num_periods
        The number of periods, at least two.
    period
        The thickness of a period, a layer of Si and a layer of Sc.
    """
    thickness_Si = (1 - fraction_Sc) * period
    thickness_Sc = fraction_Sc * period
    return dataclasses.replace(
        design,
        layers=[
            oxide,
            dataclasses.replace(top, thickness=thickness_Si - thickness_consumed),
            dataclasses.replace(Sc, thickness=thickness_Sc),
            dataclasses.replace(
                periodic,
                layers=[
                    dataclasses.replace(Si_period, thickness=thickness_Si),
                    dataclasses.replace(Sc_period, thickness=thickness_Sc),
                ],
                num_periods=num_periods - 1,
            ),
        ],
    )


def period(multilayer: optika.materials.MultilayerMirror) -> u.Quantity:
    """
    The period of a multilayer made by :func:`coating`.

    Parameters
    ----------
    multilayer
        The multilayer to measure.
    """
    return sum(layer.thickness for layer in multilayer.layers[~0].layers).to(u.nm)


def thickness(multilayer: optika.materials.MultilayerMirror) -> u.Quantity:
    """
    The total thickness of the layers of a multilayer, its oxide included.

    Parameters
    ----------
    multilayer
        The multilayer to measure.
    """
    return sum(layer.thickness for layer in multilayer.layers).to(u.nm)


def reflectivity(
    multilayer: optika.materials.MultilayerMirror,
    wavelength: u.Quantity | na.AbstractScalar,
    angle: u.Quantity | na.AbstractScalar = 0 * u.deg,
) -> na.AbstractScalar:
    """
    The reflectivity of a multilayer to unpolarized light.

    Parameters
    ----------
    multilayer
        The multilayer which reflects the light.
    wavelength
        The wavelength of the light in vacuum.
    angle
        The angle of incidence.
    """
    rays = optika.rays.RayVectorArray(
        wavelength=wavelength,
        direction=na.Cartesian3dVectorArray(np.sin(angle), 0, np.cos(angle)),
    )
    return multilayer.efficiency(
        rays=rays,
        normal=na.Cartesian3dVectorArray(0, 0, -1),
    )


print(f"{num_periods_design} periods of {period_design:.1f}, Sc fraction {fraction_Sc:.2f}")

In [ ]:
f2 = esis.flights.f2

wavelength_He_II = 256.317 * u.AA
wavelength = na.ScalarArray(
    ndarray=u.Quantity(
        [
            wavelength_He_II,
            f2.wavelength_Ne_VII,
            f2.wavelength_Si_XII,
        ]
    ).to(u.nm),
    axes="line",
)
names = ["He II", "Ne VII", "Si XII"]

instrument = f2.optics.design_single(num_distribution=0)
print(
    f"passband: {na.as_named_array(instrument.wavelength_min).ndarray.to(u.nm):.1f} "
    f"to {na.as_named_array(instrument.wavelength_max).ndarray.to(u.nm):.1f}"
)
print(f"second order of He II: {2 * wavelength_He_II.to(u.nm):.1f}")

In [ ]:
system = instrument.system
rays = system.raytrace(
    wavelength=wavelength[{"line": slice(1, None)}],
    efficiency=False,
).outputs
names_surface = [surface.name for surface in system.surfaces_all]
unvignetted = rays.unvignetted[{system.axis_surface: ~0}]


def angle_incidence(name: str) -> na.AbstractScalar:
    """
    Half the angle through which the named surface turns the rays, or NaN
    for the rays which do not reach the sensor.

    Parameters
    ----------
    name
        The name of the surface.
    """
    index = names_surface.index(name)
    before = rays.direction[{system.axis_surface: index - 1}]
    after = rays.direction[{system.axis_surface: index}]
    cos = -(before @ after) / (before.length * after.length)
    angle = (np.arccos(cos) / 2).to(u.deg)
    return np.where(unvignetted, angle, np.nan * u.deg)


angle = {name: angle_incidence(name) for name in ["primary", "grating"]}

print(
    pandas.DataFrame(
        {
            "optic": list(angle),
            "min (deg)": [np.nanmin(a).ndarray.value for a in angle.values()],
            "mean (deg)": [np.nanmean(a).ndarray.value for a in angle.values()],
            "max (deg)": [np.nanmax(a).ndarray.value for a in angle.values()],
        }
    )
    .round(2)
    .to_string(index=False)
)

In [ ]:
angle_max = max(np.nanmax(a).ndarray for a in angle.values())
ratio = reflectivity(design, wavelength, angle_max) / reflectivity(design, wavelength)

print(f"at {angle_max:.2f}, relative to normal incidence:")
for name, r in zip(names, ratio.ndarray):
    print(f"  {name}: {r:.3f}")

In [ ]:
def period_best(num_periods: int) -> u.Quantity:
    """
    The period which maximizes the geometric mean of the reflectivity at
    Ne VII and Si XII, for the given number of periods.

    Parameters
    ----------
    num_periods
        The number of periods.
    """

    def merit(period: float) -> float:
        r = reflectivity(coating(num_periods, period * u.nm), wavelength)
        return -np.sqrt(r[{"line": 1}] * r[{"line": 2}]).ndarray

    result = scipy.optimize.minimize_scalar(
        merit,
        bounds=(24, 28),
        method="bounded",
        options=dict(xatol=1e-3),
    )
    return result.x * u.nm


num_periods = list(range(2, 21))
coatings = {n: coating(n, period_best(n)) for n in num_periods}

reflectivities = na.stack(
    [reflectivity(c, wavelength) for c in coatings.values()],
    axis="num_periods",
)

In [ ]:
r_five = reflectivity(coatings[num_periods_design], wavelength)
throughput = np.square(reflectivities / r_five)
r = reflectivities

table = pandas.DataFrame(
    {
        "periods": num_periods,
        "period (nm)": [period(c).value for c in coatings.values()],
        "thickness (nm)": [thickness(c).value for c in coatings.values()],
        "R Ne VII": r[{"line": 1}].ndarray,
        "R Si XII": r[{"line": 2}].ndarray,
        "R He II": r[{"line": 0}].ndarray,
        "throughput Ne VII": throughput[{"line": 1}].ndarray,
        "throughput Si XII": throughput[{"line": 2}].ndarray,
        "(R He II / R Si XII)^2": np.square(r[{"line": 0}] / r[{"line": 2}]).ndarray,
    }
)
print(table.round(4).to_string(index=False))

In [ ]:
ratio = np.square(reflectivity(design, wavelength) / r_five)
print(f"design ({period(design):.2f}) relative to five periods of {period(coatings[num_periods_design]):.2f}:")
for name, t in zip(names, ratio.ndarray):
    print(f"  {name}: {t:.3f}")

In [ ]:
n = na.ScalarArray(np.array(num_periods), axes="num_periods")

fig, ax = plt.subplots(constrained_layout=True)
for i, color in [(1, "tab:blue"), (2, "tab:orange")]:
    na.plt.plot(
        n,
        throughput[{"line": i}],
        ax=ax,
        marker="o",
        color=color,
        label=names[i],
    )
ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
ax.set_xlabel("number of periods")
ax.set_ylabel("throughput relative to five periods")
ax.legend();

In [ ]:
wavelength_plot = na.linspace(24, 55, axis="wavelength", num=621) * u.nm

with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    for num, color in [(num_periods_design, "tab:blue"), (10, "tab:orange")]:
        na.plt.plot(
            wavelength_plot,
            reflectivity(coatings[num], wavelength_plot),
            ax=ax,
            color=color,
            label=f"{num} periods",
        )
    # label He II at the top and the two lines at the bottom, where the
    # curves are not
    for name, w, y, va in [
        (names[0], wavelength.ndarray[0], 0.97, "top"),
        (names[1], wavelength.ndarray[1], 0.03, "bottom"),
        (names[2], wavelength.ndarray[2], 0.03, "bottom"),
    ]:
        ax.axvline(w, color="gray", linestyle=":")
        ax.text(
            w,
            y,
            f" {name}",
            rotation=90,
            va=va,
            color="gray",
            transform=ax.get_xaxis_transform(),
        )
    ax.set_xlabel(f"wavelength ({ax.get_xlabel()})")
    ax.set_ylabel("reflectivity")
    ax.legend();

In [ ]:
error = na.linspace(-4, 4, axis="error", num=161) * u.percent

throughput_error = {}
for num in [num_periods_design, 10]:
    r = na.stack(
        [
            reflectivity(coating(num, period(coatings[num]) * (1 + e)), wavelength)
            for e in error.ndarray
        ],
        axis="error",
    )
    throughput_error[num] = np.square(r / r_five)

fig, axs = plt.subplots(
    ncols=2,
    sharey=True,
    constrained_layout=True,
    figsize=(8, 3.5),
)
for ax, i in zip(axs, [1, 2]):
    for num, color in [(num_periods_design, "tab:blue"), (10, "tab:orange")]:
        na.plt.plot(
            error,
            throughput_error[num][{"line": i}],
            ax=ax,
            color=color,
            label=f"{num} periods",
        )
    ax.set_title(names[i])
    ax.set_xlabel("error in the period (%)")
axs[0].set_ylabel("throughput relative to five periods")
axs[0].legend();

In [ ]:
ratio = throughput_error[10] / throughput_error[num_periods_design]
better = (ratio[{"line": slice(1, None)}] > 1).all(axis="line")
e = error.ndarray.value[better.ndarray]
print(f"ten periods beat five from {e.min():+.2f}% to {e.max():+.2f}%")

In [ ]:
def tolerance(
    throughput: na.AbstractScalar,
    fraction: float = 0.9,
) -> tuple[float, float]:
    """
    The smallest and largest errors in the period, in percent, at which the
    throughput at Ne VII and Si XII is at least the given fraction of the
    throughput with no error.

    Parameters
    ----------
    throughput
        The throughput at each line as a function of the error in the period.
    fraction
        The least acceptable fraction of the throughput with no error.
    """
    throughput = throughput[{"line": slice(1, None)}]
    nominal = throughput[{"error": error.shape["error"] // 2}]
    ok = (throughput / nominal >= fraction).all(axis="line")
    e = error.ndarray.value[ok.ndarray]
    return e.min(), e.max()


for num in [num_periods_design, 10]:
    low, high = tolerance(throughput_error[num])
    print(f"{num} periods: {low:+.2f}% to {high:+.2f}%")

In [ ]:
num = 10
spec = coatings[num]
low, high = tolerance(throughput_error[num])
r = reflectivity(spec, wavelength)
r_plot = reflectivity(spec, wavelength_plot)
peak = wavelength_plot[{"wavelength": int(r_plot.ndarray.argmax())}]

rows = [
    ("materials", "Si/Sc, Si on top"),
    ("number of periods", f"{num}"),
    ("period", f"{period(spec):.2f}"),
    (
        "tolerance on the period",
        f"{low:+.1f}% to {high:+.1f}% "
        f"({low / 100 * period(spec):+.2f} to {high / 100 * period(spec):+.2f})",
    ),
    ("Sc fraction of each period", f"{fraction_Sc:.2f}"),
    ("total thickness", f"{thickness(spec):.0f}"),
]
for name, a in angle.items():
    rows.append(
        (
            f"angle of incidence on the {name}",
            f"{np.nanmin(a).ndarray.value:.1f} to {np.nanmax(a).ndarray:.1f}",
        )
    )
rows.append(("peak of the reflectivity (model)", f"{peak.ndarray:.1f}"))
for name, w, ri in zip(names, wavelength.ndarray, r.ndarray):
    rows.append((f"reflectivity at {name} {w:.2f} (model)", f"{ri:.3f}"))

print(pandas.DataFrame(rows, columns=["parameter", "value"]).to_string(index=False))